## Experiments

Setup for every run: 50k-row subsample, stratified 5-fold CV, seed 42, same folds.
Score = mean validation F1 ± std (noise ≈ 0.01). Log every run in `experiments_log.md`.
**Keep a change only if it gains more than ~0.01**, otherwise keep the simpler version.

### ✅ Done: L2-regularized logistic regression

| max_iters | γ | λ | Threshold | F1 |
|---|---|---|---|---|
| 1000 |0.1 | 1e-4 | ≈ 0.21 | ≈ 0.43 ± 0.009 |

---

### ⬜ To do, in order of priority

**1. Compare the models**
Tune each model on its own parameters, then compare them at their best.
- least_squares (no parameter), ridge (λ), mse_gd / mse_sgd (γ), logistic (γ), reg_logistic ✅
- → keep the best model + its optimal parameters

**2. Keep more or fewer columns: `max_missing`**
Test `max_missing` ∈ {0.5, 0.8, 0.95} in `fit_preprocessing`.
- → more columns = more information, but more noise

**3. Polynomial features**
Add x², x³ of the **numerical** columns only (age, BMI, days of poor health…), not of the 0/1 columns.
Standardize the new columns with the training-fold statistics. Test degree ∈ {1, 2, 3}, then re-tune λ.
- → lets a linear model learn curved effects (risk rising faster at older ages)

**4. Missing-value flags**
Run with and without the "was missing" columns.
- → not answering a question can carry information

**5. Class imbalance (9% positives)**
Give more weight to the positives in the logistic gradient (× 5 to × 10).
- → the threshold tuning already compensates a lot, so expect a small gain

**6. Interactions and log transforms**
Products of strong features (age × general health, age × sex); log of long-tailed features
(minutes of activity, drinks per week).

**7. Remove duplicated features**
Keep one version of age, BMI, race, physical activity…
- → faster and easier to interpret, little effect on F1

**8. Final confirmation**
Re-run the best 2–3 configurations with 5-fold CV on the full data, update `run.py`
(model, parameters, threshold), submit to AIcrowd, compare with the CV estimate.

### Setup

In [1]:
import sys
sys.path.insert(0, "..")

import numpy as np
import preprocessing
import cross_validation

DATA_PATH = "../data"
SEED = 42          
N_SUB = 100000     # sub sample size
K_FOLD = 5       

### Data loading

In [2]:
## ONE TIME ONLY: read the CSV files and save the numpy arrays

# x_train, x_test, y_train, train_ids, test_ids = helpers.load_csv_data(data_path)
# np.save("../data/x_train.npy", x_train)
# np.save("../data/x_test.npy", x_test)
# np.save("../data/y_train.npy", y_train)
# np.save("../data/train_ids.npy", train_ids)
# np.save("../data/test_ids.npy", test_ids)


x_train = np.load(f"{DATA_PATH}/x_train.npy")
y_train = np.load(f"{DATA_PATH}/y_train.npy")
y = (y_train == 1).astype(int)
names = preprocessing.load_feature_names(DATA_PATH)

### rng generator - IMPORTANT (rng and calls always stay together to remain consistant)

In [3]:
rng = np.random.default_rng(SEED)                 # rng generator 

x_sub, y_sub = cross_validation.subsample(x_train, y, N_SUB, rng)
k_indices = cross_validation.build_k_indices(y_sub, K_FOLD, rng)

print(x_sub.shape, y_sub.mean(), [len(f) for f in k_indices])

(100000, 321) 0.08806 [20001, 20000, 20000, 20000, 19999]


# EXPERIMENTS (log each one) - 
To Do on bubsamples 50k samples to be faster

In [4]:
iters_list=[500, 1000, 2000]
gammas=[0.1]
lambdas=[1e-4]

results, best = cross_validation.grid_search(x_sub, y_sub, names, k_indices, "reg_logistic", lambdas, gammas, iters_list)

lambda_, gamma, max_iters, f1_mean, f1_std, t_mean = best
print(f"best: λ={lambda_:g}, γ={gamma:g}, iters={max_iters} → "
      f"F1 {f1_mean:.3f} ± {f1_std:.3f}, threshold {t_mean:.2f}")

fold 0: loss 0.2209, threshold 0.21, F1 0.418
fold 1: loss 0.2210, threshold 0.21, F1 0.413
fold 2: loss 0.2228, threshold 0.21, F1 0.438
fold 3: loss 0.2225, threshold 0.19, F1 0.445
fold 4: loss 0.2220, threshold 0.24, F1 0.430
λ=0.0001, γ=0.1, iters=500: F1 0.429 ± 0.012, threshold 0.21
fold 0: loss 0.2184, threshold 0.21, F1 0.416
fold 1: loss 0.2184, threshold 0.19, F1 0.413
fold 2: loss 0.2202, threshold 0.21, F1 0.438
fold 3: loss 0.2200, threshold 0.20, F1 0.444
fold 4: loss 0.2195, threshold 0.22, F1 0.433
λ=0.0001, γ=0.1, iters=1000: F1 0.429 ± 0.012, threshold 0.20
fold 0: loss 0.2177, threshold 0.18, F1 0.418
fold 1: loss 0.2177, threshold 0.19, F1 0.415
fold 2: loss 0.2196, threshold 0.22, F1 0.437
fold 3: loss 0.2193, threshold 0.19, F1 0.443
fold 4: loss 0.2188, threshold 0.19, F1 0.435
λ=0.0001, γ=0.1, iters=2000: F1 0.430 ± 0.011, threshold 0.19
best: λ=0.0001, γ=0.1, iters=2000 → F1 0.430 ± 0.011, threshold 0.19


In [5]:
iters_list=[1000]
gammas=[0.01, 0.05, 0.1, 0.5, 1.0]
lambdas=[1e-4]

results, best = cross_validation.grid_search(x_sub, y_sub, names, k_indices, "reg_logistic", lambdas, gammas, iters_list)

lambda_, gamma, max_iters, f1_mean, f1_std, t_mean = best
print(f"best: λ={lambda_:g}, γ={gamma:g}, iters={max_iters} → "
      f"F1 {f1_mean:.3f} ± {f1_std:.3f}, threshold {t_mean:.2f}")

fold 0: loss 0.2649, threshold 0.27, F1 0.411
fold 1: loss 0.2650, threshold 0.28, F1 0.407
fold 2: loss 0.2663, threshold 0.28, F1 0.435
fold 3: loss 0.2661, threshold 0.29, F1 0.441
fold 4: loss 0.2657, threshold 0.31, F1 0.420
λ=0.0001, γ=0.01, iters=1000: F1 0.423 ± 0.013, threshold 0.28
fold 0: loss 0.2209, threshold 0.21, F1 0.418
fold 1: loss 0.2210, threshold 0.21, F1 0.413
fold 2: loss 0.2228, threshold 0.21, F1 0.438
fold 3: loss 0.2225, threshold 0.19, F1 0.445
fold 4: loss 0.2220, threshold 0.24, F1 0.430
λ=0.0001, γ=0.05, iters=1000: F1 0.429 ± 0.012, threshold 0.21
fold 0: loss 0.2184, threshold 0.21, F1 0.416
fold 1: loss 0.2184, threshold 0.19, F1 0.413
fold 2: loss 0.2202, threshold 0.21, F1 0.438
fold 3: loss 0.2200, threshold 0.20, F1 0.444


KeyboardInterrupt: 

In [ ]:
iters_list=[1000]
gammas=[0.1]
lambdas=[0, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1]

results, best = cross_validation.grid_search(x_sub, y_sub, names, k_indices, "reg_logistic", lambdas, gammas, iters_list)

lambda_, gamma, max_iters, f1_mean, f1_std, t_mean = best
print(f"best: λ={lambda_:g}, γ={gamma:g}, iters={max_iters} → "
      f"F1 {f1_mean:.3f} ± {f1_std:.3f}, threshold {t_mean:.2f}")

fold 0: loss 0.2198, threshold 0.22, F1 0.438
fold 1: loss 0.2201, threshold 0.21, F1 0.440
fold 2: loss 0.2195, threshold 0.18, F1 0.415
fold 3: loss 0.2202, threshold 0.21, F1 0.429
fold 4: loss 0.2200, threshold 0.20, F1 0.427
λ=0, γ=0.1, iters=1000: F1 0.430 ± 0.009, threshold 0.21
fold 0: loss 0.2198, threshold 0.22, F1 0.438
fold 1: loss 0.2201, threshold 0.21, F1 0.440
fold 2: loss 0.2195, threshold 0.18, F1 0.415
fold 3: loss 0.2202, threshold 0.21, F1 0.429
fold 4: loss 0.2200, threshold 0.20, F1 0.427
λ=1e-06, γ=0.1, iters=1000: F1 0.430 ± 0.009, threshold 0.21
fold 0: loss 0.2198, threshold 0.22, F1 0.438
fold 1: loss 0.2201, threshold 0.22, F1 0.440
fold 2: loss 0.2195, threshold 0.18, F1 0.416
fold 3: loss 0.2202, threshold 0.21, F1 0.429
fold 4: loss 0.2200, threshold 0.20, F1 0.427
λ=1e-05, γ=0.1, iters=1000: F1 0.430 ± 0.009, threshold 0.21
fold 0: loss 0.2199, threshold 0.22, F1 0.438
fold 1: loss 0.2201, threshold 0.22, F1 0.439
fold 2: loss 0.2195, threshold 0.18, F1

In [ ]:
# iters_list=[500, 1000, 2000]
# gammas=[0.01, 0.05, 0.1, 0.5, 1.0]
# lambdas=[0, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1]

# results, best = cross_validation.grid_search(x_sub, y_sub, names, k_indices, "reg_logistic", lambdas, gammas, iters_list)

# lambda_, gamma, max_iters, f1_mean, f1_std, t_mean = best
# print(f"best: λ={lambda_:g}, γ={gamma:g}, iters={max_iters} → "
#       f"F1 {f1_mean:.3f} ± {f1_std:.3f}, threshold {t_mean:.2f}")

fold 0: loss 0.3284, threshold 0.38, F1 0.465
fold 1: loss 0.3281, threshold 0.36, F1 0.415
fold 2: loss 0.3282, threshold 0.38, F1 0.442
fold 3: loss 0.3282, threshold 0.37, F1 0.433
fold 4: loss 0.3283, threshold 0.33, F1 0.440
fold 5: loss 0.3282, threshold 0.41, F1 0.405
fold 6: loss 0.3283, threshold 0.40, F1 0.453
fold 7: loss 0.3282, threshold 0.37, F1 0.425
fold 8: loss 0.3283, threshold 0.36, F1 0.421
fold 9: loss 0.3282, threshold 0.37, F1 0.425
fold 10: loss 0.3283, threshold 0.35, F1 0.443
fold 11: loss 0.3280, threshold 0.33, F1 0.404
fold 12: loss 0.3283, threshold 0.36, F1 0.420
fold 13: loss 0.3284, threshold 0.36, F1 0.428
fold 14: loss 0.3281, threshold 0.35, F1 0.427
fold 15: loss 0.3281, threshold 0.33, F1 0.425
fold 16: loss 0.3283, threshold 0.41, F1 0.437
fold 17: loss 0.3282, threshold 0.38, F1 0.409
fold 18: loss 0.3282, threshold 0.31, F1 0.420
fold 19: loss 0.3283, threshold 0.36, F1 0.438
λ=0, γ=0.01, iters=500: F1 0.429 ± 0.015, threshold 0.36
fold 0: loss 

KeyboardInterrupt: 

## Differents models test


In [ ]:
params = {"max_iters": 1000, "gamma": 0.01, "lambda_": 1e-4}

f1_mean, f1_std, t_mean = cross_validation.cross_validate(x_sub, y_sub, names, k_indices, "mse_gd", params, rng=None, verbose=True)

fold 0: loss 0.0334, threshold 0.21, F1 0.429
fold 1: loss 0.0334, threshold 0.25, F1 0.420
fold 2: loss 0.0334, threshold 0.23, F1 0.422
fold 3: loss 0.0334, threshold 0.21, F1 0.413
fold 4: loss 0.0334, threshold 0.22, F1 0.421


In [ ]:
params = {"max_iters": 1000, "gamma": 0.001, "lambda_": 1e-4}

f1_mean, f1_std, t_mean = cross_validation.cross_validate(x_sub, y_sub, names, k_indices, "mse_sgd", params, rng=None, verbose=True)

fold 0: loss 0.0404, threshold 0.23, F1 0.342
fold 1: loss 0.0420, threshold 0.24, F1 0.345
fold 2: loss 0.0409, threshold 0.13, F1 0.262
fold 3: loss 0.0422, threshold 0.17, F1 0.308
fold 4: loss 0.0430, threshold 0.21, F1 0.344


In [ ]:
params = {"max_iters": 1000, "gamma": 0.1, "lambda_": 1e-4}

f1_mean, f1_std, t_mean = cross_validation.cross_validate(x_sub, y_sub, names, k_indices, "least_squares", params, rng=None, verbose=True)

fold 0: loss 0.0332, threshold 0.23, F1 0.428
fold 1: loss 0.0333, threshold 0.24, F1 0.422
fold 2: loss 0.0333, threshold 0.21, F1 0.419
fold 3: loss 0.0333, threshold 0.25, F1 0.415
fold 4: loss 0.0333, threshold 0.22, F1 0.424


In [ ]:
params = {"max_iters": 1000, "gamma": 0.1, "lambda_": 1e-4}

f1_mean, f1_std, t_mean = cross_validation.cross_validate(x_sub, y_sub, names, k_indices, "ridge", params, rng=None, verbose=True)

fold 0: loss 0.0333, threshold 0.23, F1 0.429
fold 1: loss 0.0333, threshold 0.23, F1 0.423
fold 2: loss 0.0333, threshold 0.21, F1 0.419
fold 3: loss 0.0333, threshold 0.25, F1 0.416
fold 4: loss 0.0333, threshold 0.22, F1 0.424


In [ ]:
params = {"max_iters": 1000, "gamma": 0.1, "lambda_": 1e-4}

f1_mean, f1_std, t_mean = cross_validation.cross_validate(x_sub, y_sub, names, k_indices, "logistic", params, rng=None, verbose=True)

fold 0: loss 0.2198, threshold 0.22, F1 0.438
fold 1: loss 0.2201, threshold 0.21, F1 0.440
fold 2: loss 0.2195, threshold 0.18, F1 0.415
fold 3: loss 0.2202, threshold 0.21, F1 0.429
fold 4: loss 0.2200, threshold 0.20, F1 0.427


In [ ]:
params = {"max_iters": 1000, "gamma": 0.1, "lambda_": 1e-4}

f1_mean, f1_std, t_mean = cross_validation.cross_validate(x_sub, y_sub, names, k_indices, "reg_logistic", params, rng=None, verbose=True)

fold 0: loss 0.2199, threshold 0.22, F1 0.438
fold 1: loss 0.2201, threshold 0.22, F1 0.439
fold 2: loss 0.2195, threshold 0.18, F1 0.416
fold 3: loss 0.2203, threshold 0.21, F1 0.429
fold 4: loss 0.2201, threshold 0.20, F1 0.427


In [ ]:
configs = [
    ("least_squares", {}),
    ("ridge",         {"lambda_": 1e-4}),
    ("ridge",         {"lambda_": 1e-2}),
    ("ridge",         {"lambda_": 1e-1}),
    ("mse_gd",        {"gamma": 0.001, "max_iters": 1000}),
    ("mse_gd",        {"gamma": 0.01, "max_iters": 1000}),
    ("mse_gd",        {"gamma": 0.05, "max_iters": 1000}),
    ("mse_sgd",        {"gamma": 1e-4, "max_iters": 1000}),
    ("mse_sgd",        {"gamma": 1e-3, "max_iters": 1000}),
    ("mse_sgd",        {"gamma": 1e-2, "max_iters": 1000}),
    ("mse_sgd",        {"gamma": 1e-3, "max_iters": 10000}),
    ("logistic",      {"gamma": 0.05, "max_iters": 1000}),
    ("logistic",      {"gamma": 0.1, "max_iters": 1000}),
    ("logistic",      {"gamma": 0.5, "max_iters": 1000}),
    ("reg_logistic",  {"lambda_": 1e-4, "gamma": 0.1, "max_iters": 1000}),
]
results, best = cross_validation.compare_configs(x_sub, y_sub, names, k_indices, configs, seed=SEED)

fold 0: loss 0.0329, threshold 0.22, F1 0.412
fold 1: loss 0.0329, threshold 0.22, F1 0.411
fold 2: loss 0.0331, threshold 0.23, F1 0.430
fold 3: loss 0.0331, threshold 0.24, F1 0.437
fold 4: loss 0.0330, threshold 0.23, F1 0.422
least_squares  {}                                                 F1 0.422 ± 0.010   threshold 0.228
fold 0: loss 0.0329, threshold 0.22, F1 0.413
fold 1: loss 0.0329, threshold 0.22, F1 0.410
fold 2: loss 0.0331, threshold 0.23, F1 0.431
fold 3: loss 0.0331, threshold 0.24, F1 0.437
fold 4: loss 0.0330, threshold 0.23, F1 0.422
ridge          {'lambda_': 0.0001}                                F1 0.422 ± 0.010   threshold 0.228
fold 0: loss 0.0329, threshold 0.20, F1 0.411
fold 1: loss 0.0329, threshold 0.21, F1 0.411
fold 2: loss 0.0331, threshold 0.20, F1 0.432
fold 3: loss 0.0331, threshold 0.22, F1 0.436
fold 4: loss 0.0330, threshold 0.22, F1 0.421
ridge          {'lambda_': 0.01}                                  F1 0.422 ± 0.010   threshold 0.213
fold 0:

## Train on all the data set

In [ ]:
x_train = np.load(f"{DATA_PATH}/x_train.npy")
y_train = np.load(f"{DATA_PATH}/y_train.npy")
y = (y_train == 1).astype(int)
names = preprocessing.load_feature_names(DATA_PATH)

rng = np.random.default_rng(SEED)                 # rng generator 

K_FOLD = 5
k_indices = cross_validation.build_k_indices(y, K_FOLD, rng)

In [ ]:
params = {"max_iters": 1000, "gamma": 0.1, "lambda_": 1e-4}

f1_mean, f1_std, t_mean = cross_validation.cross_validate(x_train, y, names, k_indices, "reg_logistic", params, rng=None, verbose=True)
print(f1_mean, f1_std, t_mean)

fold 0: loss 0.2210, threshold 0.21, F1 0.429
fold 1: loss 0.2211, threshold 0.19, F1 0.430
fold 2: loss 0.2205, threshold 0.20, F1 0.425
fold 3: loss 0.2210, threshold 0.19, F1 0.431
fold 4: loss 0.2211, threshold 0.21, F1 0.431


In [ ]:
print(len(k_indices), sum(len(f) for f in k_indices), len(y)) 


5 328135 328135
0.42923574336762227 0.002118384753747053 0.20177318399516908
